In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

2026-06-20 08:08:20.417217: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781942900.628908      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781942900.689353      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781942901.187611      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781942901.187662      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781942901.187665      58 computation_placer.cc:177] computation placer alr

In [2]:
df = pd.read_csv(r"/kaggle/input/datasets/vuquoctuan11/studentperformances/StudentPerformanceFactors.csv")

X = df.drop("Exam_Score", axis=1)
y = df["Exam_Score"]

X = df.drop("Exam_Score", axis=1)
y = df["Exam_Score"]

In [3]:
numerical_cols = [
    'Hours_Studied',
    'Attendance',
    'Sleep_Hours',
    'Previous_Scores',
    'Tutoring_Sessions',
    'Physical_Activity'
]

categorical_cols = [
    'Parental_Involvement',
    'Access_to_Resources',
    'Extracurricular_Activities',
    'Motivation_Level',
    'Internet_Access',
    'Family_Income',
    'Teacher_Quality',
    'School_Type',
    'Peer_Influence',
    'Learning_Disabilities',
    'Parental_Education_Level',
    'Distance_from_Home',
    'Gender'
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

X_processed = preprocessor.fit_transform(X)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X_processed,
    y,
    test_size=0.2,
    random_state=42
)

In [5]:
model = Sequential([
    Dense(256, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-06-20 08:08:32.058976: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [6]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=25,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/25
133/133 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 3.1361 - mae: 0.9046 - val_loss: 0.1263 - val_mae: 0.2070
Epoch 2/25
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0720 - mae: 0.1671 - val_loss: 0.1002 - val_mae: 0.1511
Epoch 3/25
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0514 - mae: 0.1217 - val_loss: 0.0869 - val_mae: 0.1184
Epoch 4/25
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0438 - mae: 0.1015 - val_loss: 0.0914 - val_mae: 0.1192
Epoch 5/25
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0406 - mae: 0.0943 - val_loss: 0.0910 - val_mae: 0.1103
Epoch 6/25
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0383 - mae: 0.0886 - val_loss: 0.0890 - val_mae: 0.1041
Epoch 7/25
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0375 - mae: 0.0906 - val_loss: 0.0885 - val_mae: 0.1025
Epoch 8/25
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0368 - mae: 0.0885 - val_loss: 0.0961 - val_mae: 0.1229
Epoch 9/25
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - lo

In [7]:
preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print(f"MAE: {mae:.2f}")
print(f"R²: {r2:.4f}")

42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
MAE: 0.08
R²: 0.7259


In [8]:
model.save("student_score_model.keras")

In [9]:
joblib.dump(preprocessor, "preprocessor.pkl")

['preprocessor.pkl']

In [10]:
new_student = pd.DataFrame({
    "Hours_Studied":[20],
    "Attendance":[90],
    "Parental_Involvement":["High"],
    "Access_to_Resources":["High"],
    "Extracurricular_Activities":["Yes"],
    "Sleep_Hours":[8],
    "Previous_Scores":[85],
    "Motivation_Level":["High"],
    "Internet_Access":["Yes"],
    "Tutoring_Sessions":[3],
    "Family_Income":["Medium"],
    "Teacher_Quality":["High"],
    "School_Type":["Public"],
    "Peer_Influence":["Positive"],
    "Physical_Activity":[5],
    "Learning_Disabilities":["No"],
    "Parental_Education_Level":["College"],
    "Distance_from_Home":["Near"],
    "Gender":["Male"]
})

X_new = preprocessor.transform(new_student)

pred = model.predict(X_new)

print("Predicted Exam Score:", pred[0][0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
Predicted Exam Score: 7.4566007
